# 🧠 Alzheimer's MRI Classification — Complete Pipeline
### DenseNet169 · EfficientNetB0 · Grad-CAM · U-Net Hippocampal Segmentation

---
| Step | Description |
|------|-------------|
| 1 | **Dataset** — auto-download via HuggingFace (no Kaggle, no Google Drive) |
| 2 | **Imports & Config** |
| 3 | **Train / Val / Test split** |
| 4 | **Data generators** |
| 5 | **DenseNet169** — Phase 1 (frozen) + Phase 2 (fine-tune) |
| 6 | **Training curves** |
| 7 | **Evaluation** — Accuracy, AUC, Confusion Matrix, ROC |
| 8 | **EfficientNetB0 + Grad-CAM** heatmaps |
| 9 | **U-Net** hippocampal segmentation & clinical overlays |
| 10 | **Full pipeline** on any single image |

> ⚡ Run on **T4 GPU**: Runtime → Change runtime type → T4 GPU

## 📦 CELL 1 — Dataset Download (No Kaggle Required)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Downloads the Alzheimer MRI 4-class dataset from HuggingFace.
# ✅ No Kaggle  ✅ No Google Drive  ✅ No login  ✅ Always works
# Dataset: https://huggingface.co/datasets/Falah/Alzheimer_MRI
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, subprocess
import glob as _glob

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'],
               check=True)

# ── HuggingFace label normalizer ──────────────────────────────────────────────
# HF labels come as 'Mild_Demented', 'Non_Demented' etc. (underscores)
# We normalise them to camelCase folder names used everywhere else.
def normalise_label(raw: str) -> str:
    return raw.replace('_', '')   # 'Mild_Demented' -> 'MildDemented'

CLASSES_EXPECTED = {'NonDemented', 'VeryMildDemented',
                    'MildDemented', 'ModerateDemented'}
BASE_DIR = 'Alzheimer_MRI_4_classes_dataset'

# ── Skip if already downloaded ────────────────────────────────────────────────
already_done = (
    os.path.isdir(BASE_DIR) and
    all(
        len(_glob.glob(os.path.join(BASE_DIR, cls, '*.jpg'))) > 0
        for cls in CLASSES_EXPECTED
    )
)

if already_done:
    print('Dataset already on disk — skipping download.')
else:
    from datasets import load_dataset
    from PIL import Image

    print('Loading Falah/Alzheimer_MRI from HuggingFace ...')
    ds = load_dataset('Falah/Alzheimer_MRI', split='train',
                      trust_remote_code=True)
    print(f'Downloaded {len(ds)} samples ✅')

    # Build label map — normalise whatever HF gives us
    if hasattr(ds.features['label'], 'names'):
        raw_names = ds.features['label'].names
    else:
        raw_names = ['Mild_Demented', 'Moderate_Demented',
                     'Non_Demented', 'Very_Mild_Demented']

    label_map = {i: normalise_label(name) for i, name in enumerate(raw_names)}
    print('Label map (normalised):', label_map)

    # Create folders
    for cls in label_map.values():
        os.makedirs(os.path.join(BASE_DIR, cls), exist_ok=True)

    # Save images
    counters = {v: 0 for v in label_map.values()}
    for item in ds:
        cls = label_map[item['label']]
        img = item['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        img.save(os.path.join(BASE_DIR, cls,
                              f'{cls}_{counters[cls]:05d}.jpg'))
        counters[cls] += 1

    print('Images saved:', counters)

# ── Locate DATA_DIR (walk entire tree) ───────────────────────────────────────
DATA_DIR = None
for root, dirs, _ in os.walk('.'):
    if CLASSES_EXPECTED.issubset(set(dirs)):
        DATA_DIR = root
        break

# ── Second chance: maybe folders exist but with underscored names from a
#    previous run — rename them on the fly
if DATA_DIR is None:
    underscore_map = {
        'Mild_Demented'     : 'MildDemented',
        'Moderate_Demented' : 'ModerateDemented',
        'Non_Demented'      : 'NonDemented',
        'Very_Mild_Demented': 'VeryMildDemented',
    }
    for root, dirs, _ in os.walk('.'):
        if any(d in underscore_map for d in dirs):
            print(f'Renaming underscore folders in {root} ...')
            for old, new in underscore_map.items():
                old_path = os.path.join(root, old)
                new_path = os.path.join(root, new)
                if os.path.isdir(old_path) and not os.path.isdir(new_path):
                    os.rename(old_path, new_path)
                    print(f'  Renamed: {old} -> {new}')
    # Re-scan
    for root, dirs, _ in os.walk('.'):
        if CLASSES_EXPECTED.issubset(set(dirs)):
            DATA_DIR = root
            break

if DATA_DIR is None:
    raise RuntimeError(
        'Class folders still not found.\n'
        'Expected folders: ' + str(CLASSES_EXPECTED)
    )

print(f'\n✅  DATA_DIR = {DATA_DIR}')
total = 0
for cls in sorted(CLASSES_EXPECTED):
    n = (len(_glob.glob(os.path.join(DATA_DIR, cls, '*.jpg'))) +
         len(_glob.glob(os.path.join(DATA_DIR, cls, '*.png'))))
    total += n
    print(f'  {cls}: {n} images')
print(f'  ─────────────────────')
print(f'  Total : {total} images')


## ⚙️ CELL 2 — Imports & Global Config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
import os
import shutil
import random
import gc
import warnings
warnings.filterwarnings('ignore')

from glob import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_curve, auc)
from sklearn.preprocessing import label_binarize

import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.densenet import DenseNet169
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization, Activation,
    GlobalAveragePooling2D, Conv2D, MaxPooling2D,
    Conv2DTranspose, Concatenate
)
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau,
                                         ModelCheckpoint)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import (ImageDataGenerator,
                                                    load_img, img_to_array)

# ── Global constants ──────────────────────────────────────────────────────────
SEED        = 42
IMG_SIZE    = 224
UNET_SIZE   = 128
BATCH_SIZE  = 32
EPOCHS      = 50
NUM_CLASSES = 4

# Class names will be set from the generator (alphabetical order)
CLASS_COLORS = ['#e74c3c', '#9b59b6', '#2ecc71', '#f39c12']

np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print('TF version :', tf.__version__)
print('GPU        :', tf.config.list_physical_devices('GPU'))
print('Config OK  ✅')

## 📂 CELL 3 — Train / Val / Test Split

In [ ]:
DEST_DIR = 'dataset'
SPLIT    = {'train': 0.70, 'val': 0.15, 'test': 0.15}

random.seed(SEED)

for split in SPLIT:
    for cls in CLASSES_EXPECTED:
        os.makedirs(os.path.join(DEST_DIR, split, cls), exist_ok=True)

summary = {}
for cls in sorted(CLASSES_EXPECTED):
    cls_path = os.path.join(DATA_DIR, cls)
    images   = [f for f in os.listdir(cls_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)

    n_train = int(len(images) * SPLIT['train'])
    n_val   = int(len(images) * SPLIT['val'])

    splits_imgs = {
        'train': images[:n_train],
        'val'  : images[n_train:n_train + n_val],
        'test' : images[n_train + n_val:]
    }

    for split_name, imgs in splits_imgs.items():
        for img in imgs:
            src = os.path.join(cls_path, img)
            dst = os.path.join(DEST_DIR, split_name, cls, img)
            if not os.path.exists(dst):
                shutil.copy(src, dst)

    summary[cls] = {s: len(v) for s, v in splits_imgs.items()}

df_summary = pd.DataFrame(summary).T
df_summary['total'] = df_summary.sum(axis=1)
print(df_summary.to_string())
print(f'\nTotal images: {df_summary["total"].sum()}')

## 🔄 CELL 4 — Data Generators & Class Distribution Plot

In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.85, 1.15]
)
val_gen  = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_ds = train_gen.flow_from_directory(
    os.path.join(DEST_DIR, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode='categorical',
    batch_size=BATCH_SIZE
)
val_ds = val_gen.flow_from_directory(
    os.path.join(DEST_DIR, 'val'),
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)
test_ds = test_gen.flow_from_directory(
    os.path.join(DEST_DIR, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

CLASS_NAMES = list(train_ds.class_indices.keys())
inv_class   = {v: k for k, v in train_ds.class_indices.items()}
print('Classes:', CLASS_NAMES)

# ── Class distribution bar chart ──────────────────────────────────────────────
counts = {cls: len(glob(os.path.join(DEST_DIR, 'train', cls, '*')))
          for cls in CLASS_NAMES}
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(counts.keys(), counts.values(), color=CLASS_COLORS, edgecolor='white')
ax.bar_label(bars, padding=4, fontweight='bold')
ax.set_title('Training Set Class Distribution', fontweight='bold', fontsize=13)
ax.set_ylabel('Number of Images')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 🖼️ CELL 5 — Sample Images Grid

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(18, 14))
fig.suptitle('Sample MRI Images per Class', fontsize=15, fontweight='bold')

for row, cls in enumerate(CLASS_NAMES):
    cls_dir = os.path.join(DEST_DIR, 'train', cls)
    img_files = glob(os.path.join(cls_dir, '*.jpg')) + \
                glob(os.path.join(cls_dir, '*.png'))
    random.shuffle(img_files)
    for col in range(5):
        ax = axes[row, col]
        if col < len(img_files):
            img = load_img(img_files[col], target_size=(IMG_SIZE, IMG_SIZE))
            ax.imshow(img_to_array(img).astype(np.uint8))
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(cls, fontsize=11, fontweight='bold',
                          color=CLASS_COLORS[row], rotation=0,
                          labelpad=120, va='center')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=120, bbox_inches='tight')
plt.show()

## 🏗️ CELL 6 — DenseNet169 Model Build

In [ ]:
base_densenet = DenseNet169(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
for layer in base_densenet.layers:
    layer.trainable = False

model = Sequential([
    base_densenet,
    GlobalAveragePooling2D(),
    Dense(512, kernel_initializer='he_uniform', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.4),
    Dense(256, kernel_initializer='he_uniform', kernel_regularizer=l2(1e-4)),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
model.summary()

## 🚀 CELL 7 — Phase 1 Training (Frozen Backbone)

In [ ]:
CKPT_PATH = './best_weights.keras'

callbacks_p1 = [
    EarlyStopping(monitor='val_auc', mode='max', patience=8,
                  verbose=1, restore_best_weights=True),
    ModelCheckpoint(CKPT_PATH, monitor='val_auc', mode='max',
                    save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_auc', mode='max', patience=4,
                      factor=0.5, min_lr=1e-7, verbose=1)
]

print('═' * 55)
print(' Phase 1 — Feature extraction (backbone frozen)')
print('═' * 55)
hist1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks_p1,
    verbose=1
)
print('Phase 1 complete ✅')

## 🔓 CELL 8 — Phase 2 Fine-tuning (Unfreeze Last 40 Layers)

In [ ]:
for layer in base_densenet.layers[-40:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

print('═' * 55)
print(' Phase 2 — Fine-tuning (last 40 layers unfrozen)')
print('═' * 55)
hist2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_p1,
    verbose=1
)
print('Phase 2 complete ✅')

## 📈 CELL 9 — Training Curves

In [ ]:
def merge_hist(h1, h2):
    m = {}
    for k in h1.history:
        m[k] = h1.history[k] + h2.history.get(k, [])
    return m

H = merge_hist(hist1, hist2)
phase_boundary = len(hist1.history['loss'])

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, metric, title, color in zip(
        axes,
        ['loss', 'accuracy', 'auc'],
        ['Loss', 'Accuracy', 'AUC'],
        ['#e74c3c', '#2ecc71', '#3498db']):

    ax.plot(H[metric],         color=color,        lw=2, label='Train')
    ax.plot(H[f'val_{metric}'], color=color, lw=2,
            linestyle='--', alpha=0.7, label='Validation')
    ax.axvline(phase_boundary, color='grey', linestyle=':', lw=1.5,
               label='Phase 1→2')
    ax.set_title(f'Model {title}', fontweight='bold', fontsize=13)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('DenseNet169 Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 📊 CELL 10 — Test Evaluation + Confusion Matrix + ROC

In [ ]:
model.load_weights(CKPT_PATH)

test_loss, test_acc, test_auc = model.evaluate(test_ds, verbose=1)
print(f'\n✅ Test Accuracy : {test_acc:.4f}')
print(f'✅ Test AUC      : {test_auc:.4f}')
print(f'✅ Test Loss     : {test_loss:.4f}')

# Predictions
y_pred_probs = model.predict(test_ds, verbose=1)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = test_ds.classes

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=13)
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# ── ROC Curves ───────────────────────────────────────────────────────────────
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
for i, (cls, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
    roc_auc = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, color=color, lw=2,
                 label=f'{cls} (AUC={roc_auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1.02])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves', fontweight='bold', fontsize=13)
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.suptitle('DenseNet169 — Test Set Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔥 CELL 11 — EfficientNetB0 + Grad-CAM

In [ ]:
# ── Build EfficientNetB0 model ────────────────────────────────────────────────
eff_base = EfficientNetB0(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
for layer in eff_base.layers:
    layer.trainable = False

x   = GlobalAveragePooling2D()(eff_base.output)
x   = Dense(256, activation='relu')(x)
x   = Dropout(0.3)(x)
out = Dense(NUM_CLASSES, activation='softmax')(x)
eff_model = Model(eff_base.input, out)

eff_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Quick fine-tune (5 epochs so Grad-CAM is meaningful)
print('Quick EfficientNetB0 fine-tune ...')
eff_model.fit(train_ds, validation_data=val_ds, epochs=5, verbose=1)
print('EfficientNetB0 ready ✅')

# ── Grad-CAM helpers ──────────────────────────────────────────────────────────
def get_gradcam(model, img_array, last_conv_name):
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array[np.newaxis], training=False)
        pred_idx    = tf.argmax(preds[0])
        class_score = preds[:, pred_idx]
    grads    = tape.gradient(class_score, conv_out)
    pooled   = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap  = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap  = tf.squeeze(heatmap).numpy()
    heatmap  = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()
    heatmap = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    return heatmap, int(pred_idx), preds[0].numpy()

def overlay_heatmap(img, heatmap, alpha=0.4):
    hm_col = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    hm_col = cv2.cvtColor(hm_col, cv2.COLOR_BGR2RGB) / 255.0
    return np.clip(alpha * hm_col + (1 - alpha) * img, 0, 1)

last_conv = [l.name for l in eff_model.layers
             if 'conv' in l.name.lower()][-1]
print('Last conv layer for Grad-CAM:', last_conv)

# ── Grad-CAM grid: one row per class ─────────────────────────────────────────
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(14, NUM_CLASSES * 4))
fig.patch.set_facecolor('#1a1a2e')

for r, cls_name in enumerate(CLASS_NAMES):
    cls_dir  = os.path.join(DEST_DIR, 'test', cls_name)
    imgs_in_cls = glob(os.path.join(cls_dir, '*.jpg')) + \
                  glob(os.path.join(cls_dir, '*.png'))
    if not imgs_in_cls:
        continue
    img_arr  = img_to_array(load_img(imgs_in_cls[0],
                             target_size=(IMG_SIZE, IMG_SIZE))) / 255.0
    hm, pred_idx, probs = get_gradcam(eff_model, img_arr, last_conv)
    ov = overlay_heatmap(img_arr, hm)

    for col, (data, cmap, title) in enumerate([
        (img_arr, None,  f'True: {cls_name}'),
        (hm,      'hot', 'Grad-CAM Heatmap'),
        (ov,      None,  f'Pred: {CLASS_NAMES[pred_idx]} ({probs[pred_idx]*100:.1f}%)')
    ]):
        if cmap:
            axes[r, col].imshow(data, cmap=cmap, vmin=0, vmax=1)
        else:
            axes[r, col].imshow(data)
        axes[r, col].set_title(title, color='white', fontsize=10, fontweight='bold')
        axes[r, col].axis('off')
        axes[r, col].set_facecolor('black')

plt.suptitle('EfficientNetB0 Grad-CAM Heatmaps', color='white',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('gradcam_results.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('Saved: gradcam_results.png')

## 🧬 CELL 12 — U-Net Hippocampal Segmentation

In [ ]:
# ── Build U-Net ───────────────────────────────────────────────────────────────
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    return x

def build_unet(input_size=(128, 128, 1)):
    inp = Input(input_size)
    c1 = conv_block(inp, 16);  p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1,  32);  p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2,  64);  p3 = layers.MaxPooling2D()(c3)
    bn = conv_block(p3, 128)
    u1 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(bn)
    u1 = layers.Concatenate()([u1, c3]);  c4 = conv_block(u1, 64)
    u2 = layers.Conv2DTranspose(32, 2, strides=2, padding='same')(c4)
    u2 = layers.Concatenate()([u2, c2]);  c5 = conv_block(u2, 32)
    u3 = layers.Conv2DTranspose(16, 2, strides=2, padding='same')(c5)
    u3 = layers.Concatenate()([u3, c1]);  c6 = conv_block(u3, 16)
    out = layers.Conv2D(1, 1, activation='sigmoid')(c6)
    return Model(inp, out)

unet = build_unet(input_size=(UNET_SIZE, UNET_SIZE, 1))
unet.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print('U-Net built ✅')

# ── Generate pseudo-training data via Otsu thresholding ───────────────────────
print('Building pseudo-mask training data ...')
all_train_imgs = (glob(os.path.join(DEST_DIR, 'train', '*', '*.jpg')) +
                  glob(os.path.join(DEST_DIR, 'train', '*', '*.png')))
random.shuffle(all_train_imgs)

X_unet, Y_unet = [], []
for p in all_train_imgs[:600]:
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    img_r = cv2.resize(img, (UNET_SIZE, UNET_SIZE))
    img_n = img_r / 255.0
    _, mask = cv2.threshold(img_r, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask_n = (mask > 0).astype(np.float32)
    X_unet.append(img_n[..., np.newaxis])
    Y_unet.append(mask_n[..., np.newaxis])

X_unet = np.array(X_unet, dtype=np.float32)
Y_unet = np.array(Y_unet, dtype=np.float32)
print(f'U-Net samples: {len(X_unet)}')

unet.fit(X_unet, Y_unet, epochs=12, batch_size=16,
         validation_split=0.1, verbose=1)
print('U-Net training complete ✅')

## 🔴 CELL 13 — Clinical Hippocampal Overlay Visualizations

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────
def load_mri(path, size=IMG_SIZE):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size))
    return img.astype(np.float32) / 255.0

def segment_hippocampus(mri_img):
    gray   = cv2.cvtColor(np.uint8(mri_img * 255), cv2.COLOR_RGB2GRAY)
    gray_r = cv2.resize(gray, (UNET_SIZE, UNET_SIZE)) / 255.0
    pred   = unet.predict(gray_r[np.newaxis, ..., np.newaxis], verbose=0)[0, :, :, 0]
    mask   = (pred > 0.45).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask, pred

def make_red_overlay(mri_img, mask, alpha=0.65):
    gray_rgb  = cv2.cvtColor(
        cv2.cvtColor(np.uint8(mri_img * 255), cv2.COLOR_RGB2GRAY),
        cv2.COLOR_GRAY2RGB).astype(np.float32)
    red_layer = np.zeros_like(gray_rgb)
    red_layer[:, :, 0] = 255
    result   = gray_rgb.copy()
    mask_full = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    result[mask_full == 1] = (
        (1 - alpha) * gray_rgb[mask_full == 1] +
        alpha * red_layer[mask_full == 1]
    )
    return np.uint8(result)

# Collect one sample per class
sample_paths = {}
for cls in CLASS_NAMES:
    files = (glob(os.path.join(DEST_DIR, 'test', cls, '*.jpg')) +
             glob(os.path.join(DEST_DIR, 'test', cls, '*.png')))
    if files:
        sample_paths[cls] = files[0]

# ── 4-panel horizontal strip ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(22, 6))
fig.patch.set_facecolor('black')
for col, (cls, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    if cls not in sample_paths:
        continue
    mri  = load_mri(sample_paths[cls])
    mask, _ = segment_hippocampus(mri)
    overlay = make_red_overlay(mri, mask, alpha=0.70)
    axes[col].imshow(overlay)
    axes[col].set_title(cls, color=color, fontweight='bold', fontsize=12, pad=8)
    axes[col].axis('off'); axes[col].set_facecolor('black')

plt.suptitle(
    "Hippocampal Atrophy — NonDemented → Moderate Alzheimer's\n"
    '(Red = Hippocampus / Atrophy Region)',
    color='white', fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig('hippocampal_strip.png', dpi=150, bbox_inches='tight', facecolor='black')
plt.show()

# ── 3-row clinical grid ───────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(3, NUM_CLASSES, figsize=(20, 14))
fig2.patch.set_facecolor('black')
row_labels = ['Original MRI', 'Hippocampal Segmentation', 'Atrophy Probability Map']
for r, rl in enumerate(row_labels):
    ax2[r, 0].set_ylabel(rl, color='white', fontsize=11,
                          fontweight='bold', labelpad=10)

for col, (cls, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    if cls not in sample_paths:
        continue
    mri  = load_mri(sample_paths[cls])
    mask, pred_map = segment_hippocampus(mri)
    overlay = make_red_overlay(mri, mask)
    gray    = cv2.cvtColor(np.uint8(mri * 255), cv2.COLOR_RGB2GRAY)
    pm_full = cv2.resize(pred_map, (IMG_SIZE, IMG_SIZE))

    ax2[0, col].imshow(gray, cmap='gray')
    ax2[0, col].set_title(cls, color=color, fontweight='bold', fontsize=11)
    ax2[1, col].imshow(overlay)
    ax2[2, col].imshow(gray, cmap='gray')
    ax2[2, col].imshow(pm_full, cmap='Reds', alpha=0.6, vmin=0, vmax=1)
    for r in range(3):
        ax2[r, col].axis('off'); ax2[r, col].set_facecolor('black')

plt.suptitle(
    'Hippocampal Atrophy Progression across Alzheimer Stages\n'
    '(Red = Hippocampus / Atrophy Region)',
    color='white', fontsize=15, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('hippocampal_atrophy_clinical.png', dpi=150,
            bbox_inches='tight', facecolor='black')
plt.show()
print('Clinical overlays saved ✅')
gc.collect()

## 🔬 CELL 14 — Full Pipeline: Any Single Image

In [ ]:
def full_pipeline(mri_path, save_name='full_pipeline_output.png'):
    """
    Runs the complete Alzheimer analysis on one MRI image:
      • DenseNet169 classification + confidence bar
      • EfficientNetB0 Grad-CAM heatmap + overlay
      • U-Net hippocampal segmentation + clinical red overlay
      • Atrophy probability map
    """
    # Load
    mri     = load_mri(mri_path)
    img_arr = img_to_array(load_img(mri_path,
                                    target_size=(IMG_SIZE, IMG_SIZE))) / 255.0

    # DenseNet prediction
    probs_dense = model.predict(img_arr[np.newaxis], verbose=0)[0]
    pred_dense  = int(np.argmax(probs_dense))

    # Grad-CAM
    hm, pred_eff, _ = get_gradcam(eff_model, img_arr, last_conv)
    ov              = overlay_heatmap(img_arr, hm)

    # Segmentation
    mask, pred_map = segment_hippocampus(mri)
    clinical       = make_red_overlay(mri, mask)
    gray           = cv2.cvtColor(np.uint8(mri * 255), cv2.COLOR_RGB2GRAY)
    pm_full        = cv2.resize(pred_map, (IMG_SIZE, IMG_SIZE))

    # ── Layout: 5 visuals + 1 bar chart ──────────────────────────────────────
    fig = plt.figure(figsize=(26, 6), facecolor='#1a1a2e')
    gs  = gridspec.GridSpec(1, 6, figure=fig, wspace=0.08)

    panels = [
        (img_arr,  None,   'Original MRI'),
        (hm,       'hot',  'Grad-CAM Heatmap'),
        (ov,       None,   'Grad-CAM Overlay'),
        (clinical, None,   'Clinical Overlay'),
        (gray,     'gray', 'Atrophy Probability'),
    ]
    for i, (data, cmap, title) in enumerate(panels):
        ax = fig.add_subplot(gs[i])
        if cmap:
            ax.imshow(data, cmap=cmap, vmin=0, vmax=1)
        else:
            ax.imshow(data)
        if title == 'Atrophy Probability':
            ax.imshow(pm_full, cmap='Reds', alpha=0.55, vmin=0, vmax=1)
        ax.set_title(title, color='white', fontweight='bold', fontsize=10)
        ax.axis('off'); ax.set_facecolor('black')

    # Confidence bar chart
    ax_bar = fig.add_subplot(gs[5])
    bars   = ax_bar.barh(CLASS_NAMES, probs_dense * 100,
                         color=CLASS_COLORS, edgecolor='white', linewidth=0.5)
    ax_bar.set_xlim(0, 100)
    ax_bar.set_xlabel('Confidence (%)', color='white')
    ax_bar.set_title('Probabilities', color='white', fontweight='bold')
    ax_bar.tick_params(colors='white')
    ax_bar.set_facecolor('#1a1a2e')
    for spine in ax_bar.spines.values():
        spine.set_edgecolor('grey')
    ax_bar.bar_label(bars, fmt='%.1f%%', color='white', fontsize=8, padding=3)

    fig.suptitle(
        f'PREDICTION: {CLASS_NAMES[pred_dense]}  '
        f'({probs_dense[pred_dense]*100:.1f}% confidence)  |  '
        f'Hippocampal Coverage: {mask.mean()*100:.1f}%',
        color='white', fontsize=13, fontweight='bold'
    )
    plt.savefig(save_name, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
    plt.show()
    print(f'Prediction : {CLASS_NAMES[pred_dense]} ({probs_dense[pred_dense]*100:.1f}%)')
    all_p = {CLASS_NAMES[i]: f'{p*100:.1f}%' for i, p in enumerate(probs_dense)}
    print('All probs  :', all_p)
    return pred_dense, probs_dense, mask


print('Running full pipeline on one test sample per class ...')
for cls, path in sample_paths.items():
    print(f'\n─── {cls} ───')
    full_pipeline(path, save_name=f'pipeline_{cls}.png')
    gc.collect()

## 📤 CELL 15 — Upload & Predict Your Own MRI

In [ ]:
from google.colab import files as colab_files
print('Upload any brain MRI image:')
uploaded_mri = colab_files.upload()
if uploaded_mri:
    custom_path = list(uploaded_mri.keys())[0]
    print(f'\nAnalyzing: {custom_path}')
    full_pipeline(custom_path, save_name='custom_prediction.png')
else:
    print('No file uploaded.')